# Image gate notebook

Executed by papermill in the buildspec's post_build phase, against the image
itself, to prove the ENTRYPOINT contract works before anything is pushed:
a jupyter kernel is registered, papermill can drive it, each cell's stdout
reaches the log as it finishes, and the executed copy is written out.

It is a FILE rather than a heredoc or a `python -c` inside the buildspec
because that would nest YAML, `bash -c` and `python -c` quoting three deep,
and this repo has already lost a build to buildspec quoting once.

In [ ]:
# The engines, one per analysis family in the real notebook.
import numpy as np, pandas as pd, hmmlearn, mlxtend, networkx as nx
assert np.__version__.startswith('1.'), 'numpy must be <2, got ' + np.__version__
print('GATE imports', np.__version__, pd.__version__, nx.__version__)

In [ ]:
# A groupby-heavy pandas path, which is what the real notebook is made of.
df = pd.DataFrame({'serial': ['S1', 'S1', 'S2'], 'v': [1.0, 2.0, 4.0]})
got = df.groupby('serial')['v'].max().to_dict()
assert got == {'S1': 2.0, 'S2': 4.0}, got
print('GATE pandas', got)

In [ ]:
# The degenerate-HMM path the 29-Aug run actually hit: when EM leaves a
# component unvisited, hmmlearn writes a transmat_ row of zeros and predict()
# raises. The notebook detects that and skips the serial rather than
# repairing the row. Proving the library still behaves this way here means
# the fix is exercised before a 4,549-serial run depends on it.
import warnings
from hmmlearn import hmm
warnings.filterwarnings('ignore')
rng = np.random.default_rng(13)
X = np.concatenate([rng.normal(0, 1, 400), rng.normal(6, 1, 400)]).reshape(-1, 1)
m = hmm.GaussianHMM(n_components=3, covariance_type='diag', n_iter=50,
                    min_covar=1e-3, random_state=13).fit(X)
rows = np.asarray(m.transmat_).sum(axis=1)
dead = int((~np.isclose(rows, 1.0)).sum())
assert m.means_.shape == (3, 1)
print('GATE hmm rows', np.round(rows, 3).tolist(), 'dead', dead)

In [ ]:
print('GATE OK')